In [1]:
import os
import decord
import numpy as np
import cv2
from tqdm import tqdm

In [2]:
decord.bridge.set_bridge("native")  # faster loading with native bridge

In [3]:
INPUT_DIR = r"/Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator/data/video/"
OUTPUT_DIR = r"/Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator/data/clean_video_npz/"
TARGET_FPS = 12
TARGET_SIZE = 256  # height
MAX_FRAMES = 16    # clip longer videos
SAVE_FP16 = True   # reduces storage

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
def load_video(path, target_fps):
    """Load video using Decord and normalize fps."""
    vr = decord.VideoReader(path)

    orig_fps = vr.get_avg_fps()
    if orig_fps is None or orig_fps == 0:
        orig_fps = target_fps  # fallback

    # FPS resampling indices
    step = orig_fps / target_fps
    idxs = (np.arange(0, len(vr), step)).astype(int)
    idxs = idxs[idxs < len(vr)]

    frames = vr.get_batch(idxs).asnumpy()  # (T,H,W,3) uint8
    return frames

In [6]:
def clean_frames(frames, target_size, max_frames):
    """Resize, center-crop/pad, limit frames, normalize."""
    cleaned = []

    for i, f in enumerate(frames):
        if i >= max_frames:
            break

        # Resize shorter edge to target_size
        h, w = f.shape[:2]
        scale = target_size / min(h, w)
        new_h = int(h * scale)
        new_w = int(w * scale)
        f = cv2.resize(f, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        # Center crop to (target_size, target_size)
        h, w = f.shape[:2]
        top = (h - target_size) // 2
        left = (w - target_size) // 2
        f = f[top:top+target_size, left:left+target_size]

        # Convert to float32 normalized
        f = f.astype(np.float32) / 255.0

        cleaned.append(f)

    cleaned = np.stack(cleaned, axis=0)  # (T, H, W, 3)
    return cleaned

In [7]:
def process_video(path, out_path):
    """Process one video into NPZ."""
    try:
        frames = load_video(path, TARGET_FPS)
        frames = clean_frames(frames, TARGET_SIZE, MAX_FRAMES)

        if SAVE_FP16:
            frames = frames.astype(np.float16)

        np.savez_compressed(out_path, frames=frames)
        return True

    except Exception as e:
        print(f"Error processing {path}: {e}")
        return False

In [8]:
video_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".mp4"))
][:1000]  # limit for testing

print(f"Found {len(video_files)} videos.")

Found 1000 videos.


In [10]:
video_frames = load_video(os.path.join(INPUT_DIR, video_files[0]), TARGET_FPS)

In [12]:
import torch

In [15]:
np.save("/Users/siddarthnilolkundursatish/Desktop/NYU/Courses/Computer_Vision/PhysVideoGenerator/data/sample_video_frames.npy", video_frames)

In [14]:
for v in tqdm(video_files, desc="Processing videos"):
    in_path = os.path.join(INPUT_DIR, v)
    out_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(v)[0]}.npz")
    process_video(in_path, out_path)

Processing videos: 100%|██████████| 1000/1000 [08:27<00:00,  1.97it/s]
